# Notebook used to test gencode results

In [ ]:
import numpy as np
import sys
import time
import h5py
from tqdm import tqdm

import numpy as np
import re
from math import ceil
from sklearn.metrics import average_precision_score
from torch.utils.data import Dataset
import torch
import torch.nn as nn
import pandas as pd
import matplotlib.pyplot as plt
import pickle
#import pickle5 as pickle

from sklearn.model_selection import train_test_split

from scipy.sparse import load_npz
from glob import glob

from transformers import get_constant_schedule_with_warmup
from sklearn.metrics import precision_score,recall_score,accuracy_score
import copy

from src.train import trainModel
#from src.dataloader import getData,spliceDataset,h5pyDataset,collate_fn
from src.dataloader import getData,spliceDataset,h5pyDataset,getDataPointList,getDataPointListFull,DataPointFull
from src.weight_init import keras_init
from src.losses import categorical_crossentropy_2d
from src.model import SpliceFormer
from src.evaluation_metrics import print_topl_statistics,cross_entropy_2d
from src.gpu_metrics import run_bootstrap, calculate_ap, calculate_topk
import os

In [7]:
df = pd.read_csv('../Data/transformer_40k_test_gencode_predictions_250625.gz')
df

,Y_true_acceptor,Y_pred_acceptor,Y_true_donor,Y_pred_donor
0,0.0,2.439402e-07,0.0,0.000010
1,0.0,1.543093e-07,0.0,0.000009
2,0.0,3.301046e-08,0.0,0.000002
3,0.0,2.318942e-07,0.0,0.000007
4,0.0,4.532571e-07,0.0,0.000478
...,...,...,...,...
82524995,0.0,2.839532e-05,0.0,0.000056
82524996,0.0,2.839532e-05,0.0,0.000056
82524997,0.0,2.839532e-05,0.0,0.000056
82524998,0.0,2.839532e-05,0.0,0.000056


In [8]:

if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
Y_true_acceptor = torch.as_tensor(df['Y_true_acceptor'].values, dtype=torch.int8).to(device)
Y_pred_acceptor = torch.as_tensor(df['Y_pred_acceptor'].values, dtype=torch.float32).to(device)
Y_true_donor = torch.as_tensor(df['Y_true_donor'].values, dtype=torch.int8).to(device)
Y_pred_donor = torch.as_tensor(df['Y_pred_donor'].values, dtype=torch.float32).to(device)


# Compute scores
ap_score = calculate_ap(Y_true_acceptor, Y_pred_acceptor, Y_true_donor, Y_pred_donor, device, device)
topk_score = calculate_topk(Y_true_acceptor, Y_pred_acceptor, Y_true_donor, Y_pred_donor)

print(f"Average Precision: {ap_score:.4f}")
print(f"Top-k Accuracy: {topk_score:.4f}")

Average Precision: 0.0007
Top-k Accuracy: 0.0052


In [43]:
import gffutils

# Replace with your GTF path
gtf_path = 'gencode.v24.annotation.gtf'

db_path = 'gencode_v24.db'

if not os.path.isfile(db_path): 
    gffutils.create_db(gtf_path, db_path, force=True, disable_infer_genes=True, disable_infer_transcripts=True)


db = gffutils.FeatureDB(db_path)

In [69]:
input_file = 'canonical_dataset.txt'
output_file = 'gencode_test.txt'

allowed_chroms = {"chr1", "chr3", "chr5", "chr7", "chr9"}

with open(input_file, "r") as fin, open(output_file, "w") as fout:
    # Read header and write it out
    # header = fin.readline()
    # fout.write(header)

    for line in fin:
        fields = line.strip().split("\t")
        if len(fields) < 8:
            continue  # Skip malformed lines

        name, paralog, chrom = fields[0], fields[1], fields[2]

        # Apply filters
        if paralog != "0":
            continue
        if chrom not in allowed_chroms:
            continue

        fout.write(line)


In [9]:
SL=5000
CL_max=40000
NUM_ACCUMULATION_STEPS=1
BATCH_SIZE = 16
data_dir = '../Data/gencode_40k_dataset_test_.h5'
h5f = h5py.File(data_dir)
for key in h5f.keys():
    print(key)
num_idx = len(h5f.keys())//2
all_indices = list(range(num_idx))

test_dataset = h5pyDataset(h5f,list(range(num_idx)))


X0
X1
X10
X11
X12
X13
X14
X15
X16
X2
X3
X4
X5
X6
X7
X8
X9
Y0
Y1
Y10
Y11
Y12
Y13
Y14
Y15
Y16
Y2
Y3
Y4
Y5
Y6
Y7
Y8
Y9


In [3]:
SL=5000
CL_max=40000
NUM_ACCUMULATION_STEPS=1
BATCH_SIZE = 16
data_dir = '../Data/gencode_40k_dataset_test_.h5'


h5f = h5py.File(data_dir)

num_idx = len(h5f.keys())//2

test_dataset = h5pyDataset(h5f,list(range(num_idx)))
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=0)

temp = 1
n_models = 10
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
model_m = SpliceFormer(CL_max,bn_momentum=0.01/NUM_ACCUMULATION_STEPS,depth=4,heads=4,n_transformer_blocks=2,determenistic=True)
model_m.apply(keras_init)
model_m = model_m.to(device)

if torch.cuda.device_count() > 1:
    model_m = nn.DataParallel(model_m)
model_m = nn.DataParallel(model_m)

output_class_labels = ['Null', 'Acceptor', 'Donor']

#for output_class in [1,2]:
models = [copy.deepcopy(model_m) for i in range(n_models)]
[model.load_state_dict(torch.load('../Results/PyTorch_Models/transformer_encoder_40k_171022_{}'.format(i),map_location=device)) for i,model in enumerate(models)]

for model in models:
    model.eval()
    
Y_true_acceptor, Y_pred_acceptor = [],[]
Y_true_donor, Y_pred_donor = [],[]
ce_2d = []

for (batch_chunks,target_chunks) in tqdm(test_loader):
    batch_chunks = torch.transpose(batch_chunks[0].to(device),1,2)
    target_chunks = torch.transpose(torch.squeeze(target_chunks[0].to(device),0),1,2)
    #print(np.max(target_chunks.cpu().numpy()[:,2,:]))
    n_chunks = int(np.ceil(batch_chunks.shape[0]/BATCH_SIZE))
    batch_chunks = torch.chunk(batch_chunks, n_chunks, dim=0)
    target_chunks = torch.chunk(target_chunks, n_chunks, dim=0)
    targets_list = []
    outputs_list = []
    for j in range(len(batch_chunks)):
        batch_features = batch_chunks[j]
        targets = target_chunks[j]
        outputs = ([models[i](batch_features)[0].detach() for i in range(n_models)])
        #outputs = (outputs[0]+outputs[1]+outputs[2]+outputs[3]+outputs[4])/n_models
        outputs = torch.mean(torch.stack(outputs),dim=0)
        #outputs = odds_gmean(torch.stack(outputs))
        #outputs = (outputs[0]+outputs[1]+outputs[2])/n_models
        targets_list.extend(targets.unsqueeze(0))
        outputs_list.extend(outputs.unsqueeze(0))

    targets = torch.transpose(torch.vstack(targets_list),1,2).cpu().numpy()
    outputs = torch.transpose(torch.vstack(outputs_list),1,2).cpu().numpy()
    ce_2d.append(cross_entropy_2d(targets,outputs))

    is_expr = (targets.sum(axis=(1,2)) >= 1)
    Y_true_acceptor.extend(targets[is_expr, :, 1].flatten())
    Y_true_donor.extend(targets[is_expr, :, 2].flatten())
    Y_pred_acceptor.extend(outputs[is_expr, :, 1].flatten())
    Y_pred_donor.extend(outputs[is_expr, :, 2].flatten())


  0%|          | 0/17 [02:25<?, ?it/s]


KeyboardInterrupt: 